In [ ]:
!pip install openai-whisper transformers sentencepiece docx2txt python-docx fpdf pydub matplotlib nltk gTTS

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

import whisper
from pydub import AudioSegment
from gtts import gTTS
import matplotlib.pyplot as plt
import numpy as np
from docx import Document
from docx.shared import Inches
from transformers import MarianMTModel, MarianTokenizer
import zipfile, os
from google.colab import files

uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
print(f"Uploaded file: {audio_file}")

model = whisper.load_model("small")  # or "base" for faster processing
result = model.transcribe(audio_file)
full_text = result["text"]
print("Transcribed text:\n", full_text[:500])

from nltk.tokenize import sent_tokenize
sentences = sent_tokenize(full_text)
print(f"Found {len(sentences)} sentences.")

pause = AudioSegment.silent(duration=10000)  # 10 sec pause
audio_with_pauses = AudioSegment.silent(duration=0)
sentence_timestamps = []
current_time = 0

for i, sent in enumerate(sentences, 1):
    tts = gTTS(sent)
    tts.save(f"sent_{i}.mp3")
    seg = AudioSegment.from_mp3(f"sent_{i}.mp3")
    audio_with_pauses += seg + pause
    sentence_timestamps.append((current_time, current_time + len(seg), sent))
    current_time += len(seg) + len(pause)

audio_with_pauses.export("output_with_pauses.wav", format="wav")

plt.figure(figsize=(14, 5))
plt.plot(audio_with_pauses.get_array_of_samples(), alpha=0.7, color='tab:blue')
plt.title("Audio with pauses (full signal)")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.savefig("wave_full.png", dpi=300)
plt.show()

plt.figure(figsize=(14, 6))
samples = np.array(audio_with_pauses.get_array_of_samples())
plt.plot(samples, alpha=0.7, color='tab:orange')
for i, (start, end, _) in enumerate(sentence_timestamps, 1):
    x = int(start * audio_with_pauses.frame_rate / 1000)
    plt.axvline(x, color='red', linestyle='--', alpha=0.7)
    plt.text(x, samples.max() * 0.9 if len(samples) > 0 else 1, str(i), color='blue',
             fontsize=10, rotation=90, ha='center', va='bottom')

plt.title("Audio with sentence segmentation (red lines + numbers)")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.savefig("wave_sentences.png", dpi=300)
plt.show()

translated_sentences = []
try:
    model_name = 'Helsinki-NLP/opus-mt-en-uk'
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model_mt = MarianMTModel.from_pretrained(model_name)
    for sent in sentences:
        batch = tokenizer([sent], return_tensors="pt", padding=True)
        translated = model_mt.generate(**batch)
        ukr_sent = tokenizer.decode(translated[0], skip_special_tokens=True)
        translated_sentences.append(ukr_sent)
except Exception as e:
    print("⚠️ Translation failed:", e)
    translated_sentences = sentences  # fallback

doc = Document()
doc.add_heading("Audio Handout & Transcript Report", 0)

# Page 1: Full text & Full Waveform
doc.add_heading("Page 1: Full Text & Signal Overview", level=1)
doc.add_paragraph(full_text)
if os.path.exists("wave_full.png"):
    doc.add_paragraph()
    doc.add_picture("wave_full.png", width=Inches(6.5))

# Page 2: Numbered sentences with timestamps & Segmented Waveform
doc.add_page_break()
doc.add_heading("Page 2: Sentence Segmentation & Timestamps", level=1)
for i, (start, end, sent) in enumerate(sentence_timestamps, 1):
    doc.add_paragraph(f"{i}. [{start/1000:.1f}s - {end/1000:.1f}s] {sent}")

if os.path.exists("wave_sentences.png"):
    doc.add_paragraph()
    doc.add_picture("wave_sentences.png", width=Inches(6.5))

# Page 3: Translation (EN->UKR)
doc.add_page_break()
doc.add_heading("Page 3: Translation (EN -> UKR)", level=1)
for i, sent in enumerate(translated_sentences, 1):
    doc.add_paragraph(f"{i}. {sent}")

doc.save("Handout.docx")

with open("sentence_timestamps.txt", "w") as f:
    for i, (start, end, sent) in enumerate(sentence_timestamps, 1):
        f.write(f"{i}. [{start/1000:.1f}s - {end/1000:.1f}s] {sent}\n")

zip_filename = "Handout_Files.zip"
with zipfile.ZipFile(zip_filename, "w") as zipf:
    for fname in ["output_with_pauses.wav", "wave_full.png", "wave_sentences.png",
                  "Handout.docx", "sentence_timestamps.txt"]:
        if os.path.exists(fname):
            zipf.write(fname)

files.download(zip_filename)